# Launching a Nextflow pipeline

The big one. You'll install Nextflow, configure it for Google Cloud
Batch, and run an nf-core pipeline end to end. We're using
`nf-core/viralrecon` here because it fits paired-end viral
whole-genome sequencing, which is what the H1N1 demo data is.

**Time estimates:**

- **First run (cold)**: 30-45 minutes. Most of the wall clock is GCP
  Batch allocating Spot VMs and each VM pulling the container image
  it needs (viralrecon uses ~15 different containers). Containers
  get cached on the work bucket after the first pull.
- **Subsequent runs (cached)**: 10-15 minutes. Reusing cached
  containers is fast; the only delay is Spot VM allocation and the
  actual computational work.

Cost is cents, not dollars. The `test` profile runs on a tiny
synthetic dataset with minimal GCP Batch resources.

> **Kernel:** When JupyterLab prompts "Select Kernel," pick **`Python 3 (Local)`** (under "Start python Kernel"). Don't pick `Python 3 (ipykernel) (Local)`, PyTorch, or TensorFlow — those are missing libraries this notebook needs and will fail at the first GCS read with `ModuleNotFoundError`. If you already see `Python 3 (Local)` in the top-right of this tab, you're good to go.

## Tools you'll use

This notebook combines Nextflow (workflow engine), nf-core (pipeline
catalog), and GCP Batch (compute backend) to run viralrecon end-to-end.
If any of those names are new, [05-reference.ipynb](05-reference.ipynb)
has a paragraph on each plus the rationale for the pinned versions
this notebook uses.

## Why viralrecon for this data?

`nf-core/viralrecon` is built for paired-end short-read viral
whole-genome sequencing. The H1N1 demo dataset is exactly that:
Illumina paired-end FASTQs from Influenza A samples (SRR13000260,
261, 262, each with `_1`/`_2` pairs).

Different data (RNA-seq, bacterial genomics, metagenomics) just means
a different pipeline name. The launch mechanics here transfer
directly. Browse [the full nf-core catalog](https://nf-co.re/pipelines)
to pick something else.

In [ ]:
import subprocess

DETECTED_PROJECT = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True
).strip()

# Most values auto-detected from the Workbench environment.
PROJECT_ID    = DETECTED_PROJECT                              # @param {type:"string"}
REGION        = "us-central1"                                 # @param {type:"string"}

# Buckets — defaults follow APGAP's naming pattern. Override if needed.
# Auto-detected after we list visible buckets below.
WORK_BUCKET    = ""                                           # @param {type:"string"}
RESULTS_BUCKET = ""                                           # @param {type:"string"}

# Custom VPC network + subnet. APGAP projects don't have a default VPC,
# so Batch needs to be told which network to put VMs on. Defaults below
# are for the H1N1 project. For other projects, ask your lead for the
# project's network + subnet names and override here.
NETWORK        = "h1n1-network-rt8r"                          # @param {type:"string"}
SUBNETWORK     = "h1n1-subnetwork-rt8r"                       # @param {type:"string"}

# Which nf-core pipeline + which profile + which release tag.
PIPELINE_REPO     = "nf-core/viralrecon"                      # @param {type:"string"}
PIPELINE_REVISION = "2.6.0"                                   # @param {type:"string"}
PROFILE           = "test,gcb"                                # @param {type:"string"}
# `test` = nf-core's built-in tiny synthetic dataset
# `gcb`  = the GCP Batch profile we'll write in cell 10
# Pinning the revision matters: the pipeline's main branch sometimes
# references config files that aren't yet shipped. A tagged release is
# always self-consistent.

In [ ]:
import gcsfs
import subprocess

active_account = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"], text=True
).strip()
print(f"GCP project: {PROJECT_ID}")
print(f"Region:      {REGION}")
print(f"Identity:    {active_account}\n")

fs = gcsfs.GCSFileSystem()
all_buckets = fs.ls("")

if not RESULTS_BUCKET:
    out = [b.rstrip("/") for b in all_buckets if "seqera-output" in b]
    if len(out) > 1:
        print(f"⚠ Multiple seqera-output buckets visible ({len(out)}): {out}")
        print(f"  Auto-selecting the first one. Override RESULTS_BUCKET above if that's wrong.")
    if out:
        RESULTS_BUCKET = f"gs://{out[0]}/"
        print(f"Auto-selected RESULTS_BUCKET: {RESULTS_BUCKET}")
if not WORK_BUCKET:
    # Use the same bucket with a /nf-work/ prefix to keep work staging separate
    WORK_BUCKET = RESULTS_BUCKET.rstrip("/") + "/nf-work/"
    print(f"Auto-selected WORK_BUCKET:    {WORK_BUCKET}")

# Auto-detect the project's custom VPC. APGAP projects don't have the
# default GCP VPC; they use one custom network per project (e.g.
# `h1n1-network-rt8r`). The service account can list SUBNETS but typically
# not NETWORKS (missing compute.networks.list), so query subnets first and
# extract the parent network from each row's network field. Only fall back
# to a networks listing if the subnet query returned nothing AND NETWORK
# wasn't already set in the parameter cell. Pattern courtesy of Glen
# Otero's launch notebook.
def _detect_network_subnet(network, subnet):
    if not subnet:
        r = subprocess.run(
            ["gcloud", "compute", "networks", "subnets", "list",
             f"--project={PROJECT_ID}",
             f"--regions={REGION}",
             "--format=value(name,network)",
             "--limit=1"],
            capture_output=True, text=True,
        )
        rows = r.stdout.strip().splitlines()
        if rows:
            parts = rows[0].split()
            subnet = parts[0]
            if not network and len(parts) > 1:
                # network field is a full URL; bare name is the last segment.
                network = parts[1].rsplit("/", 1)[-1]
    if not network:
        r = subprocess.run(
            ["gcloud", "compute", "networks", "list",
             f"--project={PROJECT_ID}",
             "--format=value(name)",
             "--limit=1"],
            capture_output=True, text=True,
        )
        if r.stdout.strip():
            network = r.stdout.strip().splitlines()[0]
    return network, subnet

NETWORK, SUBNETWORK = _detect_network_subnet(NETWORK, SUBNETWORK)
print(f"Network:      {NETWORK or '(none)'}")
print(f"Subnetwork:   {SUBNETWORK or '(none)'}")

if not NETWORK or not SUBNETWORK:
    raise RuntimeError(
        "Couldn't auto-detect a VPC + subnet, and NETWORK or SUBNETWORK is "
        "empty in the parameter cell above. Ask your project lead for the "
        "network and subnet names for this project, then paste them into "
        "NETWORK and SUBNETWORK above and re-run. "
        f"(Got NETWORK={NETWORK!r}, SUBNETWORK={SUBNETWORK!r}.)"
    )

## Installing Nextflow and Java

Nextflow runs on the JVM. We install both Nextflow and JDK 17 via whichever conda-compatible package manager the Workbench has on PATH (`mamba`, `micromamba`, or `conda` — the underlying image varies). Versions are pinned (Nextflow 23.10 LTS, JDK 17, viralrecon 2.6.0); see [05-reference.ipynb](05-reference.ipynb) for what each pin is working around.

The install is a no-op if those versions are already there. Otherwise it upgrades or downgrades to match.

If the install fails (network glitch, bioconda channel hiccup), just re-run the cell. If it keeps failing, drop into a terminal and run `<pkg_manager> install -y -c bioconda -c conda-forge openjdk=17 nextflow=23.10` yourself, swapping in whichever of `mamba`, `micromamba`, or `conda` is present.

In [ ]:
import shutil
import subprocess

# Workbench images vary: older ones ship with `conda`, newer ones with
# `micromamba`. Pick whichever conda-compatible manager is on PATH.
for _mgr in ("mamba", "micromamba", "conda"):
    if shutil.which(_mgr):
        PKG_MANAGER = _mgr
        break
else:
    raise RuntimeError(
        "No conda-compatible package manager (mamba, micromamba, conda) "
        "found in PATH. This notebook needs one to install Nextflow + JDK "
        "from the bioconda / conda-forge channels."
    )

print(f"Pinning Nextflow 23.10 + JDK 17 via {PKG_MANAGER} (no-op if already correct)...")
subprocess.run(
    [PKG_MANAGER, "install", "-y", "-c", "bioconda", "-c", "conda-forge",
     "openjdk=17", "nextflow=23.10"],
    check=True,
)
print("Install complete.")

In [ ]:
import subprocess

print("=== java -version ===")
subprocess.run(["java", "-version"], check=True)
print("\n=== nextflow -version ===")
subprocess.run(["nextflow", "-version"], check=True)

## Pointing Nextflow at GCP Batch

Nextflow needs four pieces of information:

- The executor to use. `google-batch` submits each pipeline step as a
  GCP Batch job.
- The GCP project and region to run jobs in.
- A staging bucket for intermediate work. Nextflow writes per-step
  inputs and outputs here as the pipeline runs.
- A service account. We default to the Workbench's own service
  account, which Seqera uses for this project too, so the permissions
  are already in place.

All of this goes into a `nextflow.config` file. The next cell writes
it; the one after that prints it back out so you can see what landed
on disk.

In [ ]:
config_content = f"""
profiles {{
    gcb {{
        // Merge ALL process-scope settings into ONE block.
        // Never mix dot-syntax (process.executor = ...) with a block-syntax
        // process {{ ... }} in the same profile — Nextflow's legacy parser
        // silently discards the dot-syntax assignment when it then sees a
        // block. The first time we hit this, the executor silently fell
        // back to 'local' and the pipeline failed on the GCS work directory.
        process {{
            executor      = 'google-batch'
            cpus          = 2
            memory        = '4 GB'
            // Retry transient Batch failures (VM preemption, brief network
            // hiccups) up to twice before giving up on the task.
            errorStrategy = {{ task.attempt <= 2 ? 'retry' : 'finish' }}
            maxRetries    = 2
        }}

        google {{
            project  = '{PROJECT_ID}'
            location = '{REGION}'
            batch {{
                serviceAccountEmail = '{active_account}'
                // APGAP projects use a custom VPC rather than the default
                // network. Batch needs to know which network and subnet
                // to attach VMs to. Auto-detected above; override the
                // NETWORK / SUBNETWORK params if the wrong ones are picked.
                network    = 'projects/{PROJECT_ID}/global/networks/{NETWORK}'
                subnetwork = 'projects/{PROJECT_ID}/regions/{REGION}/subnetworks/{SUBNETWORK}'
                // Spot VMs cut cost ~70% but can be preempted. The
                // errorStrategy retry above handles the occasional
                // preemption. Set to false if you need guaranteed
                // completion (e.g., overnight production runs).
                spot = true
            }}
        }}

        // Where Nextflow stages intermediate files.
        workDir = '{WORK_BUCKET}'
    }}
}}
"""

with open("nextflow.config", "w") as f:
    f.write(config_content)

print("Wrote nextflow.config")

In [ ]:
with open("nextflow.config") as f:
    print(f.read())

## Step 1: dry run with `-preview`

Before sending anything to GCP Batch (which costs real money), ask
Nextflow to parse the pipeline and our config without actually running
steps. If there's a syntax error in the config or Nextflow can't
resolve the pipeline, we find out in ten seconds instead of ten
minutes.

In [ ]:
import subprocess

result = subprocess.run(
    ["nextflow", "run", PIPELINE_REPO,
     "-r", PIPELINE_REVISION,
     "-profile", PROFILE,
     "-preview",
     "-c", "nextflow.config",
     # No trailing slash: Nextflow appends its own / when joining the
     # outdir with per-step relative paths, and a trailing / here produces
     # a malformed `gs://bucket/path//step/file` that breaks publishFile.
     "--outdir", RESULTS_BUCKET.rstrip("/") + "/viralrecon-test-results"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("DRY RUN FAILED — fix config before continuing:")
    print(result.stderr)

## Step 2: actually run the test pipeline

Runs `nf-core/viralrecon` on the synthetic test dataset bundled with
the `test` profile. Each pipeline step becomes a GCP Batch job. As
steps complete, you'll see Nextflow's progress table update inline.

Expected duration: five to seven minutes.

Expected cost: a few cents.

Want to watch what's happening over on GCP? Open
https://console.cloud.google.com/batch/jobs in a new tab while it
runs.

In [ ]:
import subprocess

# We use `-resume` so if you re-run this cell, Nextflow will reuse cached
# results from any steps that already completed.
import time

process = subprocess.Popen(
    ["nextflow", "run", PIPELINE_REPO,
     "-r", PIPELINE_REVISION,
     "-profile", PROFILE,
     "-c", "nextflow.config",
     "-resume",
     # No trailing slash: see comment in the dry-run cell above.
     "--outdir", RESULTS_BUCKET.rstrip("/") + "/viralrecon-test-results"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

# Stream output while Nextflow is alive. We use poll() rather than
# `for line in process.stdout` because Nextflow's JVM spawns Batch
# polling threads that keep the stdout pipe open even after the main
# process exits — that combination makes iter-readline hang
# indefinitely waiting for an EOF that never arrives.
try:
    while process.poll() is None:
        line = process.stdout.readline()
        if line:
            print(line, end="", flush=True)
        else:
            time.sleep(0.5)  # no data yet but still running; avoid busy-loop
    # Drain anything still buffered after Nextflow has exited.
    remaining = process.stdout.read()
    if remaining:
        print(remaining, end="", flush=True)
finally:
    process.stdout.close()

print(f"\nNextflow exited with code {process.returncode}")

## What just happened

Nextflow pulled `nf-core/viralrecon` from GitHub. (Subsequent runs
reuse the cached copy.)

For each step in the pipeline it submitted a GCP Batch job, waited
for the job to finish, and pulled the outputs back.

Intermediate files went into `{WORK_BUCKET}`. Usually safe to delete
after a successful run; see the cleanup section below.

Final outputs landed in the results bucket from `--outdir`. For a map
of what viralrecon writes where, see [05-reference.ipynb](05-reference.ipynb).

The full Nextflow log is in `.nextflow.log` in your notebook
directory. An HTML execution report sits in `<results>/pipeline_info/`.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
# Mirror the no-trailing-slash convention used in --outdir on the launch
# command so listing matches what Nextflow actually wrote.
results_prefix = RESULTS_BUCKET.replace("gs://", "").rstrip("/") + "/viralrecon-test-results"

try:
    entries = fs.ls(results_prefix)
    print(f"Contents of gs://{results_prefix}/ ({len(entries)} items):")
    for e in sorted(entries):
        info = fs.info(e)
        kind = "DIR " if info.get("type") == "directory" else "FILE"
        size_mb = info.get("size", 0) / 1024 / 1024
        print(f"  [{kind}] {size_mb:>7.1f} MB  {e.rsplit('/', 1)[-1]}")
except FileNotFoundError:
    print("Results bucket prefix not found. Did the pipeline run complete successfully?")

## The MultiQC report

`nf-core/viralrecon` includes a MultiQC step that aggregates QC
metrics from every other step into a single self-contained HTML
report. That's the first thing to look at after a run finishes.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
# MultiQC report location depends on the pipeline; viralrecon puts it under
# multiqc/.../multiqc_report.html
multiqc_candidates = [
    p for p in fs.find(results_prefix)
    if p.endswith("multiqc_report.html")
]

if multiqc_candidates:
    report_path = multiqc_candidates[0]
    print(f"MultiQC report: gs://{report_path}")
    print(f"\nTo view: open `gs://{report_path}` in the JupyterLab GCS browser")
    print("(left sidebar), right-click → Download, then open the local copy.")
else:
    print("No multiqc_report.html found yet — pipeline may still be running.")

## Variant calls and consensus sequences

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()

vcfs = [p for p in fs.find(results_prefix) if p.endswith((".vcf", ".vcf.gz"))]
consensus = [p for p in fs.find(results_prefix) if p.endswith(".consensus.fa")]

def _show(label, paths, limit=5):
    print(f"{label} ({len(paths)}):")
    for p in paths[:limit]:
        size_kb = fs.info(p).get("size", 0) / 1024
        print(f"  {size_kb:>7.1f} KB  gs://{p}")
    if len(paths) > limit:
        print(f"  ... and {len(paths) - limit} more")

_show("VCF files", vcfs)
print()
_show("Consensus FASTA files", consensus)

## Running on your real data

Until now we've used the `test` profile, which uses synthetic data.
To run viralrecon on your actual dataset (the FASTQs in your
analytical-dataset bucket), two things change:

1. Build a samplesheet. It's a CSV that tells viralrecon which FASTQ
   pairs belong to which sample. The next cell builds one from your
   `DATASET_URI`. For the samplesheet format reference, see
   [05-reference.ipynb](05-reference.ipynb).
2. Launch with the samplesheet instead of the `test` profile. The
   cell after that shows the command, but doesn't run it. A real
   viralrecon run on real-size FASTQs takes hours and costs real
   money. Read the command, edit if needed, and run it on purpose.

In [ ]:
import gcsfs
import re
import pandas as pd

# Use the same DATASET_URI auto-detection as notebook 02.
DATASET_URI = ""  # @param {type:"string"}  -- paste a dataset URI here to override

fs = gcsfs.GCSFileSystem()
if not DATASET_URI:
    all_analytical = [b.rstrip("/") for b in fs.ls("") if "analytical-dataset" in b]
    pathogen = PROJECT_ID.split("-")[0]
    matched = [b for b in all_analytical if b.startswith(pathogen)]
    candidates = matched or all_analytical
    if not candidates:
        raise RuntimeError("No analytical-dataset bucket visible — set DATASET_URI manually.")
    DATASET_URI = f"gs://{candidates[0]}/"
    print(f"Auto-selected dataset for samplesheet: {DATASET_URI}")

dataset_path = DATASET_URI.replace("gs://", "").rstrip("/")
all_files = sorted(fs.ls(dataset_path))
fastqs = [f for f in all_files if f.endswith(".fastq.gz")]

# Group _1/_2 pairs by sample stem.
pairs = {}
for f in fastqs:
    m = re.search(r"/([^/]+)_([12])\.fastq\.gz$", f)
    if m:
        stem, direction = m.group(1), m.group(2)
        pairs.setdefault(stem, {})[direction] = f"gs://{f}"

# Build the viralrecon samplesheet format:
# sample,fastq_1,fastq_2
rows = []
incomplete = []
for sample, files in sorted(pairs.items()):
    if "1" in files and "2" in files:
        rows.append({
            "sample": sample,
            "fastq_1": files["1"],
            "fastq_2": files["2"],
        })
    else:
        missing = "_2" if "1" in files else "_1"
        incomplete.append(f"{sample} (missing {missing})")

if incomplete:
    print(f"⚠ Skipped {len(incomplete)} sample(s) missing a paired read:")
    for name in incomplete:
        print(f"  - {name}")
    print()

samplesheet = pd.DataFrame(rows)
samplesheet.to_csv("samplesheet.csv", index=False)
print(f"Wrote samplesheet.csv with {len(samplesheet)} samples:\n")
print(samplesheet.to_string(index=False))

### Optional: enrich the samplesheet with APGAP metadata

The samplesheet we just wrote has the three columns viralrecon
expects: `sample`, `fastq_1`, `fastq_2`. The next cell loads APGAP's
`_apgap_metadata.csv` if it's in the dataset bucket, and shows you
the same samples with pathogen, collection facility, instrument, and
so on joined on.

Important: the enriched view is for your reference only. The actual
`samplesheet.csv` we hand to Nextflow stays as the minimal three
columns. viralrecon would ignore the extras anyway.

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
metadata_path = DATASET_URI.replace("gs://", "").rstrip("/") + "/_apgap_metadata.csv"

try:
    with fs.open(metadata_path, "r") as f:
        metadata = pd.read_csv(f)
    # APGAP metadata is per-FILE (R1 and R2 are separate rows). For an
    # enriched samplesheet view we collapse to per-sample by picking the R1
    # row's values; sample-level fields (collection date, lab, etc.) are
    # the same for both directions anyway.
    metadata["sample_stem"] = metadata["filename"].str.replace(
        r"_[12]\.fastq\.gz$", "", regex=True
    )
    r1_metadata = (
        metadata[metadata["filename"].str.endswith("_1.fastq.gz")]
        .set_index("sample_stem")
    )
    enriched = samplesheet.merge(
        r1_metadata, left_on="sample", right_index=True, how="left",
    )
    preferred = [c for c in [
        "sample", "fastq_1", "fastq_2",
        "SAMPLE ID", "PATHOGEN/ORGANISM NAME (OR METAGENOMIC)",
        "BIOSPECIMEN TYPE", "DATE COLLECTED",
        "SEQUENCING INSTRUMENT MAKE AND MODEL",
        "SEQUENCING LAB (ORIGINATING LAB)", "SOURCE LOCATION STATE",
    ] if c in enriched.columns]
    print(f"Enriched view ({len(enriched)} samples):\n")
    print(enriched[preferred].to_string(index=False))
except FileNotFoundError:
    print(f"No _apgap_metadata.csv found in {DATASET_URI} — skipping enrichment.")
    print("(The samplesheet.csv we wrote above still works for launching.)")

## Real-data launch (commented out)

The next cell would launch viralrecon on the samplesheet we just
built. It's commented out on purpose. Uncomment, read what you're
about to run, then run it only when you actually mean to.

Expected duration: hours, depending on how many samples and how big
each one is.

Expected cost: real money. Tens to hundreds of dollars depending on
the data. If your project's budget is sensitive, check with your lead
before running.

In [ ]:
# import subprocess
# subprocess.run([
#     "nextflow", "run", PIPELINE_REPO,
#     "-r", PIPELINE_REVISION,
#     "-profile", "gcb",       # NOTE: just "gcb", not "test,gcb"
#     "-c", "nextflow.config",
#     "-resume",
#     "--input", "samplesheet.csv",
#     "--outdir", RESULTS_BUCKET.rstrip("/") + "/viralrecon-real-results",  # no trailing /
#     "--platform", "illumina",
#     "--protocol", "metagenomic",  # or 'amplicon' if you used a primer scheme
#     "--genome", "MN908947.3",     # SARS-CoV-2 default. For Influenza A H1N1
#                                   # try "FJ966082.1" or pass a custom --fasta
#                                   # + --gff pair. See nf-co.re/viralrecon docs.
# ], check=True)

## Cleaning up

After a successful run you can delete the intermediate work
directory to save on storage. The final outputs in `--outdir` are
untouched.

This cell is commented out too. Uncomment when you're sure you won't
want to `-resume` from this run later. Deleting the work directory
destroys the cache `-resume` reads from, so any re-run after cleanup
starts fresh from step 1.

In [ ]:
# import subprocess
# subprocess.run(["gsutil", "-m", "rm", "-r", WORK_BUCKET], check=True)
# print(f"Deleted work directory {WORK_BUCKET}")

## Watching runs on GCP

- GCP Batch jobs console: https://console.cloud.google.com/batch/jobs
  (pick your project from the top-bar picker)
- Cloud Storage browser: https://console.cloud.google.com/storage
- Seqera workspace (if your project has one) is linked from the
  project detail page in the APGAP web app

## Where to go next

- Build a custom samplesheet with different groupings or extra
  metadata: see the [viralrecon usage docs](https://nf-co.re/viralrecon/usage)
- Try a different pipeline: browse the [nf-core catalog](https://nf-co.re/pipelines)
- Prepare more input data: [04-download-from-public-repos.ipynb](04-download-from-public-repos.ipynb)
- Back to the [getting-started overview](01-getting-started.ipynb)